# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show metadata information
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}\n")
print(f"Version: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Explore available record sets
print("Available record sets (@id and name):\n")

for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id}, name: {rs.name}")

# For demonstration, get the first record set and print its fields
if metadata.record_sets:
    record_set = metadata.record_sets[0]
    print(f"\nFields for RecordSet '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

    # For further cells
    record_set_id = record_set.id

else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

To stay consistent, we use variable `record_set_id` which is the `@id` of the record set we chose above.

In [ ]:
dataframes = {}

# If multiple record sets, extract all
record_set_ids = [rs.id for rs in metadata.record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Print out column names for chosen record set
print(f"Available columns in record set '@id': {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())

dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Choose a numeric field for analysis. Edit accordingly to match a known numeric field `@id` from the overview.
# For this example, let's guess likely numeric field names commonly used in proteomics datasets.

numeric_candidate_fields = ['cr:field:unique_peptides', 'cr:field:peptide_count', 'cr:field:abundance', 'cr:field:MW', 'cr:field:coverage']
selected_field_id = None
for f in numeric_candidate_fields:
    if f in dataframes[record_set_id].columns:
        selected_field_id = f
        break

if selected_field_id is None:
    # Try to pick numeric looking column
    for col in dataframes[record_set_id].columns:
        # Heuristic: select on numeric dtype or plausible proteomics metric
        if dataframes[record_set_id][col].dtype in ['int64', 'float64']:
            selected_field_id = col
            break

if selected_field_id is not None:
    print(f"Selected numeric field for analysis: {selected_field_id}\n")
    try:
        col_values = pd.to_numeric(dataframes[record_set_id][selected_field_id], errors='coerce')
    except Exception as e:
        print(f"Error: {e}")
        col_values = dataframes[record_set_id][selected_field_id]
    
    threshold = col_values.mean() if col_values.notnull().any() else 0
    filtered_df = dataframes[record_set_id][col_values > threshold].copy()
    print(f"Filtered records with {selected_field_id} > {threshold} (mean):\n")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{selected_field_id}_normalized"] = (col_values - col_values.mean()) / col_values.std()
    print(f"Normalized {selected_field_id} for filtered records:\n")
    print(filtered_df[[selected_field_id, f"{selected_field_id}_normalized"]].head())

    # Guess grouping field
    candidate_groups = ['cr:field:accession', 'cr:field:gene', 'cr:field:id', 'cr:field:description']
    group_field = None
    for cg in candidate_groups:
        if cg in filtered_df.columns:
            group_field = cg
            break

    if group_field is None and len(filtered_df.columns) > 1:
        # Take first non-numeric
        for col in filtered_df.columns:
            if not (filtered_df[col].dtype == 'int64' or filtered_df[col].dtype == 'float64'):
                group_field = col
                break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (mean statistics):\n")
        print(grouped_df.head())
else:
    print("No numeric field available for analysis in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_field_id is not None:
    # Plot distribution of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[record_set_id][selected_field_id].dropna().astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {selected_field_id}")
    plt.xlabel(selected_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If normalized field exists, plot that too
    norm_col = f"{selected_field_id}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[norm_col].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of normalized {selected_field_id}")
        plt.xlabel(norm_col)
        plt.ylabel("Frequency")
        plt.show()

    # Boxplot by group if available
    if group_field is not None and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        top_n = filtered_df[group_field].value_counts().head(10).index
        sns.boxplot(x=group_field, y=selected_field_id, data=filtered_df[filtered_df[group_field].isin(top_n)])
        plt.title(f"{selected_field_id} by {group_field} (top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field selected; skipping visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Croissant dataset was loaded and record sets, fields, and data structure were reviewed using `mlcroissant`.
- We extracted and explored the main data record set and performed basic normalization and grouping.
- Data distributions and summary statistics were visualized for the main numeric field.

**Next steps**: Refine analysis procedures to focus on biologically meaningful fields (such as protein abundances, peptide counts, or normalization metrics), integrate with domain-specific visualizations, or join with other omics data as needed.